# KolektorSDD2 — Phase 2 baseline review

Reads the report produced by `scripts/train_baseline.py`.
Re-run that script if `reports/baseline/metrics.json` is missing.

Problem statement: see `PROBLEM.md`.

In [ ]:
from pathlib import Path
import json
from IPython.display import Image, Markdown, display

ROOT = Path("..").resolve()
REPORT_DIR = ROOT / "reports" / "baseline"
report = json.loads((REPORT_DIR / "metrics.json").read_text(encoding="utf-8"))
print("Generated UTC:", report["generated_at_utc"])
print("Device:", report["device"])
print("Best epoch:", report["best_epoch"], "val PR-AUC", report["best_val_pr_auc"])

## Splits

Validation is stratified from official train only. Official test is frozen.

In [ ]:
display(report["split_counts"])
sel = report["threshold_selection"]
print("Threshold rule:", sel["rule"])
print("Chosen threshold:", sel["threshold"])

## Metrics at the val-chosen threshold

Primary metric: defective-class recall on official test. Accuracy is misleading at ~10.7% prevalence.

In [ ]:
rows = []
for role, m in report["metrics_at_chosen_threshold"].items():
    cm = m["confusion_matrix"]
    rows.append(
        {
            "split": role,
            "accuracy": round(m["accuracy"], 4),
            "precision_def": round(m["precision_defective"], 4),
            "recall_def": round(m["recall_defective"], 4),
            "f1_def": round(m["f1_defective"], 4),
            "pr_auc": None if m["pr_auc_defective"] is None else round(m["pr_auc_defective"], 4),
            "tp": cm["tp"],
            "fp": cm["fp"],
            "fn": cm["fn"],
            "tn": cm["tn"],
        }
    )
display(rows)

In [ ]:
for name in ("history.png", "pr_curve_val.png", "pr_curve_test.png", "confusion_matrices.png"):
    path = REPORT_DIR / name
    if path.is_file():
        display(Markdown(f"**{name}**"))
        display(Image(filename=str(path)))